In [1]:
import numpy as np
import torch

In [2]:
reviews = [
    "Amazing movie, really enjoyed it.",
    "Great story and excellent acting.",
    "Loved every moment of it.",
    "Very entertaining and enjoyable.",
    "The movie was fantastic.",
    "Brilliant acting and direction.",
    "A fun movie with a great story.",
    "Really impressive and emotional.",
    "One of the best movies I’ve seen.",
    "Highly enjoyable from start to finish.",
    "Very boring and slow.",
    "The story was disappointing.",
    "Poor acting and weak direction.",
    "I didn’t enjoy the movie.",
    "Too predictable and dull.",
    "The movie felt unnecessarily long.",
    "Weak story with poor execution.",
    "Not interesting at all.",
    "A complete waste of time.",
    "The movie was disappointing"
]

labels = [1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0]


In [3]:
MAX_LENGTH = 10
vocabulary = {
    '<PAD>':0
}

In [4]:
for review in reviews:
    for word in review.split():
        if word not in vocabulary:
            vocabulary[word] = len(vocabulary)

In [5]:
vocabulary

{'<PAD>': 0,
 'Amazing': 1,
 'movie,': 2,
 'really': 3,
 'enjoyed': 4,
 'it.': 5,
 'Great': 6,
 'story': 7,
 'and': 8,
 'excellent': 9,
 'acting.': 10,
 'Loved': 11,
 'every': 12,
 'moment': 13,
 'of': 14,
 'Very': 15,
 'entertaining': 16,
 'enjoyable.': 17,
 'The': 18,
 'movie': 19,
 'was': 20,
 'fantastic.': 21,
 'Brilliant': 22,
 'acting': 23,
 'direction.': 24,
 'A': 25,
 'fun': 26,
 'with': 27,
 'a': 28,
 'great': 29,
 'story.': 30,
 'Really': 31,
 'impressive': 32,
 'emotional.': 33,
 'One': 34,
 'the': 35,
 'best': 36,
 'movies': 37,
 'I’ve': 38,
 'seen.': 39,
 'Highly': 40,
 'enjoyable': 41,
 'from': 42,
 'start': 43,
 'to': 44,
 'finish.': 45,
 'boring': 46,
 'slow.': 47,
 'disappointing.': 48,
 'Poor': 49,
 'weak': 50,
 'I': 51,
 'didn’t': 52,
 'enjoy': 53,
 'movie.': 54,
 'Too': 55,
 'predictable': 56,
 'dull.': 57,
 'felt': 58,
 'unnecessarily': 59,
 'long.': 60,
 'Weak': 61,
 'poor': 62,
 'execution.': 63,
 'Not': 64,
 'interesting': 65,
 'at': 66,
 'all.': 67,
 'complete'

### Encode all reviews

In [6]:
def encode(review):
    tokens = [vocabulary[word] for word in review.split()]
    while len(tokens)<10:
        tokens.append(0)
    return tokens

In [7]:
encodeed_reviews=[]
for review in reviews:
    encodeed_reviews.append(encode(review))

In [8]:
encodeed_reviews

[[1, 2, 3, 4, 5, 0, 0, 0, 0, 0],
 [6, 7, 8, 9, 10, 0, 0, 0, 0, 0],
 [11, 12, 13, 14, 5, 0, 0, 0, 0, 0],
 [15, 16, 8, 17, 0, 0, 0, 0, 0, 0],
 [18, 19, 20, 21, 0, 0, 0, 0, 0, 0],
 [22, 23, 8, 24, 0, 0, 0, 0, 0, 0],
 [25, 26, 19, 27, 28, 29, 30, 0, 0, 0],
 [31, 32, 8, 33, 0, 0, 0, 0, 0, 0],
 [34, 14, 35, 36, 37, 38, 39, 0, 0, 0],
 [40, 41, 42, 43, 44, 45, 0, 0, 0, 0],
 [15, 46, 8, 47, 0, 0, 0, 0, 0, 0],
 [18, 7, 20, 48, 0, 0, 0, 0, 0, 0],
 [49, 23, 8, 50, 24, 0, 0, 0, 0, 0],
 [51, 52, 53, 35, 54, 0, 0, 0, 0, 0],
 [55, 56, 8, 57, 0, 0, 0, 0, 0, 0],
 [18, 19, 58, 59, 60, 0, 0, 0, 0, 0],
 [61, 7, 27, 62, 63, 0, 0, 0, 0, 0],
 [64, 65, 66, 67, 0, 0, 0, 0, 0, 0],
 [25, 68, 69, 14, 70, 0, 0, 0, 0, 0],
 [18, 19, 20, 71, 0, 0, 0, 0, 0, 0]]

### Define the Dataset

In [9]:
from torch.utils.data import Dataset
class SentimentDataset(Dataset):
    def __init__(self):
        self.x = torch.tensor(encodeed_reviews,dtype=torch.long)
        self.y = torch.tensor(labels,dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self,index):
        return self.x[index],self.y[index]

In [10]:
dataset = SentimentDataset()

### Create the dataloader

In [11]:
from torch.utils.data import DataLoader
loader = DataLoader(dataset=dataset,batch_size=2,shuffle=True)

### Define The Model Class

In [12]:
class SentimentAnalysis(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.embeddings = torch.nn.Embedding(
            num_embeddings=len(vocabulary),
            embedding_dim=8
        )
        self.lstm = torch.nn.LSTM(
            input_size = 8,
            hidden_size = 16,
            batch_first = True
        )

        self.linear = torch.nn.Linear(16,1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self,x):
        x = self.embeddings(x)
        output,(hidden,cell) = self.lstm(x)
        x = hidden[-1]
        x = self.linear(x)
        x=self.sigmoid(x)
        return x.squeeze(-1)

### Detect the GPU

In [13]:
device = ''
if torch.cuda.is_available():
    device='cuda'
else:
    device='cpu'

In [14]:
device

'cuda'

In [15]:
model = SentimentAnalysis()
model = model.to(device)


In [16]:
loss_function = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)
epochs = 50

### Training Loop

In [17]:
for epoch in range(epochs):
    total_loss = 0
    for x,y in loader:
        x = x.to(device)
        y = y.to(device)
        predictions = model(x)
        loss = loss_function(predictions,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(f"epoch : {epoch} loss : {loss}")

epoch : 0 loss : 0.6907570362091064
epoch : 1 loss : 0.7361754179000854
epoch : 2 loss : 0.7286202311515808
epoch : 3 loss : 0.7513415813446045
epoch : 4 loss : 0.7410486340522766
epoch : 5 loss : 0.7547271251678467
epoch : 6 loss : 0.6962929964065552
epoch : 7 loss : 0.6194261312484741
epoch : 8 loss : 0.2538045644760132
epoch : 9 loss : 0.15272986888885498
epoch : 10 loss : 0.18970733880996704
epoch : 11 loss : 0.08576154708862305
epoch : 12 loss : 0.08881663531064987
epoch : 13 loss : 0.07498008012771606
epoch : 14 loss : 0.09626299142837524
epoch : 15 loss : 0.1256365180015564
epoch : 16 loss : 1.2229441404342651
epoch : 17 loss : 1.1737836599349976
epoch : 18 loss : 1.2853584289550781
epoch : 19 loss : 0.1033467948436737
epoch : 20 loss : 0.0732169970870018
epoch : 21 loss : 0.0336916521191597
epoch : 22 loss : 0.06957405060529709
epoch : 23 loss : 0.05618082731962204
epoch : 24 loss : 0.09022900462150574
epoch : 25 loss : 0.0581914484500885
epoch : 26 loss : 0.08693680167198181
e

In [21]:
def predict(review):
    model.eval()
    encodeed_review = torch.tensor([encode(review)],dtype=torch.long).to(device)
    with torch.no_grad():
        prediction = model(encodeed_review)
        print(prediction)
        if prediction>0.5:
            print("Positive")
        else:
            print("Negative")

In [22]:
predict("Brilliant entertaining movie ")

tensor([0.9966], device='cuda:0')
Positive


In [24]:
predict("didn’t enjoy")

tensor([0.0989], device='cuda:0')
Negative


In [25]:
predict("Not interesting")

tensor([0.0990], device='cuda:0')
Negative
